In [3]:
import os
import requests
import time

os.makedirs("mvp", exist_ok=True)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

years = list(range(2005, 2026))
url_start = "https://www.basketball-reference.com/awards/awards_{}.html"

for year in years:
    filepath = f"mvp/{year}.html"
    
    if os.path.exists(filepath):
        print(f"Skipping {year}, already exists")
        continue

    url = url_start.format(year)
    r = requests.get(url, headers=headers, timeout=10)

    if r.status_code == 429 or "Rate Limited Request" in r.text:
        print(f"Rate limited on {year}, stopping to avoid re-block")
        break

    if r.status_code != 200:
        print(f"Failed {year}: status {r.status_code}")
        continue

    with open(filepath, "w+", encoding="utf-8") as f:
        f.write(r.text)

    print(f"Saved {year}")
    time.sleep(6)

print("Done!")

Saved 2005
Saved 2006
Saved 2007
Saved 2008
Saved 2009
Saved 2010
Saved 2011
Saved 2012
Saved 2013
Saved 2014
Saved 2015
Saved 2016
Saved 2017
Saved 2018
Saved 2019
Saved 2020
Saved 2021
Saved 2022
Saved 2023
Saved 2024
Saved 2025
Done!


In [9]:
from bs4 import BeautifulSoup

with open("mvp/2020.html", encoding="utf-8") as f:
    page = f.read()

In [10]:
soup = BeautifulSoup(page, "html.parser")

In [11]:
soup.find('tr', class_="over_header").decompose()

In [13]:
mvp_table = soup.find(id="mvp")

In [19]:
from io import StringIO
import pandas as pd

In [20]:
mvp_2020 = pd.read_html(StringIO(str(mvp_table)))[0]

In [21]:
mvp_2020

,Rank,Player,Age,Tm,First,Pts Won,Pts Max,Share,G,MP,PTS,TRB,AST,STL,BLK,FG%,3P%,FT%,WS,WS/48
0,1,Giannis Antetokounmpo,25,MIL,85,962,1010,0.952,63,30.4,29.5,13.6,5.6,1.0,1.0,0.553,0.304,0.633,11.1,0.279
1,2,LeBron James,35,LAL,16,753,1010,0.746,67,34.6,25.3,7.8,10.2,1.2,0.5,0.493,0.348,0.693,9.8,0.204
2,3,James Harden,30,HOU,0,367,1010,0.363,68,36.5,34.3,6.6,7.5,1.8,0.9,0.444,0.355,0.865,13.1,0.254
3,4,Luka DonÄiÄ,20,DAL,0,200,1010,0.198,61,33.6,28.8,9.4,8.8,1.0,0.2,0.463,0.316,0.758,8.8,0.207
4,5,Kawhi Leonard,28,LAC,0,168,1010,0.166,57,32.4,27.1,7.1,4.9,1.8,0.6,0.470,0.378,0.886,8.7,0.226
5,6,Anthony Davis,26,LAL,0,82,1010,0.081,62,34.4,26.1,9.3,3.2,1.5,2.3,0.503,0.330,0.846,11.1,0.250
6,7,Chris Paul,34,OKC,0,26,1010,0.026,70,31.5,17.6,5.0,6.7,1.6,0.2,0.489,0.365,0.907,8.9,0.193
7,8,Damian Lillard,29,POR,0,23,1010,0.023,66,37.5,30.0,4.3,8.0,1.1,0.3,0.463,0.401,0.888,11.6,0.225
8,9,Nikola JokiÄ,24,DEN,0,18,1010,0.018,73,32.0,19.9,9.7,7.0,1.2,0.6,0.528,0.314,0.817,9.8,0.202
9,10,Pascal Siakam,25,TOR,0,17,1010,0.017,60,35.2,22.9,7.3,3.5,1.0,0.9,0.453,0.359,0.792,5.4,0.123


In [30]:
dfs = []
for year in years:
    with open("mvp/{}.html".format(year), encoding="utf-8") as f:
        page = f.read()
    soup = BeautifulSoup(page, "html.parser")
    soup.find('tr', class_="over_header").decompose()
    mvp_table = soup.find(id="mvp")
    mvp = pd.read_html(StringIO(str(mvp_table)))[0]
    mvp["year"] = year

    dfs.append(mvp)

In [31]:
mvps = pd.concat(dfs)

In [32]:
mvps.head()

,Rank,Player,Age,Tm,First,Pts Won,Pts Max,Share,G,MP,...,TRB,AST,STL,BLK,FG%,3P%,FT%,WS,WS/48,year
0,1,Steve Nash,30,PHO,65,1066,1270,0.839,75,34.3,...,3.3,11.5,1.0,0.1,0.502,0.431,0.887,10.9,0.203,2005
1,2,Shaquille O'Neal,32,MIA,58,1032,1270,0.813,73,34.1,...,10.4,2.7,0.5,2.3,0.601,NaN,0.461,11.0,0.211,2005
2,3,Dirk Nowitzki,26,DAL,0,349,1270,0.275,78,38.7,...,9.7,3.1,1.2,1.5,0.459,0.399,0.869,15.6,0.248,2005
3,4,Tim Duncan,28,SAS,1,328,1270,0.258,66,33.4,...,11.1,2.7,0.7,2.6,0.496,0.333,0.670,11.2,0.245,2005
4,5,Allen Iverson,29,PHI,2,240,1270,0.189,75,42.3,...,4.0,7.9,2.4,0.1,0.424,0.308,0.835,9.0,0.136,2005


In [33]:
mvps.tail()

,Rank,Player,Age,Tm,First,Pts Won,Pts Max,Share,G,MP,...,TRB,AST,STL,BLK,FG%,3P%,FT%,WS,WS/48,year
7,7T,Anthony Edwards,23,MIN,0,12,1000,0.012,79,36.3,...,5.7,4.5,1.2,0.6,0.447,0.395,0.837,8.4,0.140,2025
8,9,Stephen Curry,36,GSW,0,2,1000,0.002,70,32.2,...,4.4,6.0,1.1,0.4,0.448,0.397,0.933,7.9,0.168,2025
9,10T,Jalen Brunson,28,NYK,0,1,1000,0.001,65,35.4,...,2.9,7.3,0.9,0.1,0.488,0.383,0.821,8.3,0.172,2025
10,10T,James Harden,35,LAC,0,1,1000,0.001,79,35.3,...,5.8,8.7,1.5,0.7,0.410,0.352,0.874,8.3,0.143,2025
11,10T,Evan Mobley,23,CLE,0,1,1000,0.001,71,30.5,...,9.3,3.2,0.9,1.6,0.557,0.370,0.725,9.0,0.200,2025


In [34]:
mvps.to_csv("mvps.csv")

In [42]:
import os
os.makedirs("player", exist_ok=True)

In [43]:
player_stats_url = "https://www.basketball-reference.com/leagues/NBA_{}_per_game.html"

url = player_stats_url.format(2020)
data = requests.get(url, headers=headers, timeout=10)
with open("player/2020.html", "w+", encoding="utf-8") as f:
    f.write(data.text)

In [96]:
from selenium import webdriver

driver = webdriver.Chrome()

In [89]:
with open("player/{}.html".format(year), "w+", encoding="utf-8") as f:
    f.write(html)

In [87]:
import os
os.makedirs("player", exist_ok=True)  # ADD: folder must exist first

for year in years:
    filepath = f"player/{year}.html"
    if os.path.exists(filepath):        # ADD: skip already-saved years
        print(f"Skipping {year}")
        continue

    url = player_stats_url.format(year)

    driver.get(url)
    driver.execute_script("window.scrollTo(1,10000)")
    time.sleep(2)

    html = driver.page_source
    with open(filepath, "w+", encoding="utf-8") as f:   # ADD: encoding="utf-8"
        f.write(html)

    print(f"Saved {year}")   # ADD: so you can see progress

print("Done!")

Skipping 2005
Skipping 2006
Skipping 2007
Skipping 2008
Skipping 2009
Skipping 2010
Skipping 2011
Skipping 2012
Skipping 2013
Skipping 2014
Skipping 2015
Skipping 2016
Skipping 2017
Skipping 2018
Skipping 2019
Skipping 2020
Skipping 2021
Skipping 2022
Skipping 2023
Skipping 2024
Skipping 2025
Done!


In [100]:
import os
from bs4 import BeautifulSoup
import pandas as pd

dfs = []
for year in years:
    with open("player/{}.html".format(year), encoding="utf-8") as f:
        page = f.read()

    soup = BeautifulSoup(page, "html.parser")
    for row in soup.find_all('tr', class_="thead"):
        row.decompose()

    player_table = soup.find(id="per_game_stats")
    player = pd.read_html(str(player_table))[0]
    player["Year"] = year
    dfs.append(player)

players = pd.concat(dfs)
players.to_csv("players.csv", index=False)
players

C:\Users\acer\AppData\Local\Temp\ipykernel_3852\3888827873.py:15: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  player = pd.read_html(str(player_table))[0]
C:\Users\acer\AppData\Local\Temp\ipykernel_3852\3888827873.py:15: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  player = pd.read_html(str(player_table))[0]
C:\Users\acer\AppData\Local\Temp\ipykernel_3852\3888827873.py:15: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  player = pd.read_html(str(player_table))[0]
C:\Users\acer\AppData\Local\Temp\ipykernel_3852\3888827873.py:15: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a f

,Rk,Player,Age,Team,Pos,G,GS,MP,FG,FGA,...,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards,Year
0,1.0,Allen Iverson,29.0,PHI,PG,75.0,75.0,42.3,10.3,24.2,...,3.3,4.0,7.9,2.4,0.1,4.6,1.9,30.7,"MVP-5,DPOY-11,AS,NBA1",2005
1,2.0,Kobe Bryant,26.0,LAL,SG,66.0,66.0,40.7,8.7,20.1,...,4.5,5.9,6.0,1.3,0.8,4.1,2.6,27.6,"AS,NBA3",2005
2,3.0,LeBron James,20.0,CLE,SF,80.0,80.0,42.4,9.9,21.1,...,6.0,7.4,7.2,2.2,0.7,3.3,1.8,27.2,"MVP-6,AS,NBA2",2005
3,4.0,Dirk Nowitzki,26.0,DAL,PF,78.0,78.0,38.7,8.5,18.5,...,8.5,9.7,3.1,1.2,1.5,2.3,2.8,26.1,"MVP-3,AS,NBA1",2005
4,5.0,Amar'e Stoudemire,22.0,PHO,C,80.0,80.0,36.1,9.3,16.7,...,6.2,8.9,1.6,1.0,1.6,2.4,3.5,26.0,"MVP-9,AS,NBA2",2005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
731,566.0,Jahlil Okafor,29.0,IND,C,1.0,0.0,3.0,0.0,0.0,...,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,NaN,2025
732,567.0,Zyon Pullin,23.0,MEM,SG,3.0,0.0,1.0,0.0,0.3,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,2025
733,568.0,Isaiah Stevens,24.0,MIA,PG,3.0,0.0,2.0,0.0,0.7,...,0.7,0.7,0.0,0.3,0.0,0.0,0.0,0.0,NaN,2025
734,569.0,Terry Taylor,25.0,SAC,PF,3.0,0.0,2.0,0.0,0.3,...,0.0,0.3,0.7,0.0,0.0,0.0,0.0,0.0,NaN,2025


In [102]:
players.to_csv("players.csv")

In [103]:
team_stats_url = "https://www.basketball-reference.com/leagues/NBA_{}_standings.html"

In [104]:
for year in years:
    url = team_stats_url.format(year)
    
    data = requests.get(url)
    
    with open("team/{}.html".format(year), "w+",encoding="utf-8") as f:
        f.write(data.text)

In [123]:
dfs = []
for year in years:
    with open("team/{}.html".format(year), encoding="utf-8") as f:
        page = f.read()
        
    soup = BeautifulSoup(page, "html.parser")
    soup.find('tr', class_="thead").decompose()
    team_table = soup.find(id="divs_standings_E")
    team = pd.read_html(str(team_table))[0]
    team["Year"] = year
    team["Team"] = team["Eastern Conference"]
    del team["Eastern Conference"]
    dfs.append(team)
    
    soup = BeautifulSoup(page, "html.parser")
    soup.find('tr', class_="thead").decompose()
    team_table = soup.find(id="divs_standings_W")
    team = pd.read_html(str(team_table))[0]
    team["Year"] = year
    team["Team"] = team["Western Conference"]
    del team["Western Conference"]
    dfs.append(team)

C:\Users\acer\AppData\Local\Temp\ipykernel_3852\1419540365.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  team = pd.read_html(str(team_table))[0]
C:\Users\acer\AppData\Local\Temp\ipykernel_3852\1419540365.py:18: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  team = pd.read_html(str(team_table))[0]
C:\Users\acer\AppData\Local\Temp\ipykernel_3852\1419540365.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  team = pd.read_html(str(team_table))[0]
C:\Users\acer\AppData\Local\Temp\ipykernel_3852\1419540365.py:18: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version.

In [124]:
teams = pd.concat(dfs)

In [125]:
teams

,W,L,W/L%,GB,PS/G,PA/G,SRS,Year,Team
0,45,37,.549,—,101.3,100.4,0.35,2005,Boston Celtics*
1,43,39,.524,2.0,99.1,99.9,-1.07,2005,Philadelphia 76ers*
2,42,40,.512,3.0,91.4,92.9,-1.82,2005,New Jersey Nets*
3,33,49,.402,12.0,99.7,101.4,-1.81,2005,Toronto Raptors
4,33,49,.402,12.0,97.3,99.7,-2.72,2005,New York Knicks
...,...,...,...,...,...,...,...,...,...
13,52,30,.634,—,114.3,109.8,4.97,2025,Houston Rockets*
14,48,34,.585,4.0,121.7,116.9,4.79,2025,Memphis Grizzlies*
15,39,43,.476,13.0,114.2,115.4,-0.74,2025,Dallas Mavericks
16,34,48,.415,18.0,113.9,116.7,-2.45,2025,San Antonio Spurs


In [128]:
teams.to_csv("teams.csv")